In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from wordcloud import WordCloud

In [2]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
df = pd.read_csv("portals.csv")
df.head()

,name,title,url,author,publisher,issued,publisher_classification,description,tags,license_id,...,place,location,country,language,status,metadatacreated,generator,api_endpoint,api_type,full_metadata_download
0,a2gov_org,"Ann Arbor, Michigan",https://www.a2gov.org/services/data-catalog/,City of Ann Arbor,City of Ann Arbor,NaN,NaN,City of Ann Arbor's Open Data Catalog (USA),ctic unitedstates,NaN,...,"Ann Arbor, Michigan","42.2681569,-83.7312291",US,en,active,2011-06-27T18:12:57.439Z,NaN,NaN,NaN,NaN
1,acikveri-sahinbey-bel-tr,Açık Veri Portali - Test Yayını,https://acikveri.sahinbey.bel.tr/dataset,pinardag,SahinBey Belediyesi,2015-01-31,Government,The first official open data portal of Turkey,turkey national,Unknown,...,"Gaziantep,Turkey","37.0587715,37.380137",TR,tr,active,NaN,NaN,NaN,NaN,NaN
2,africa_open_data,Africa Open Data,https://africaopendata.org/,Africa Open Data,Africa Open Data,NaN,NaN,Africa's largest central repository for Govern...,ckan africa,NaN,...,Africa,"2.0000003,15.9999997",AF,en,active,2013-03-15T07:17:26.251Z,CKAN: 2.1.3,https://africaopendata.org/api/,NaN,NaN
3,ajuntament-de-tarragona,Open Data Tarragona,https://opendata.tarragona.cat/,Ajuntament de Tarragona,Ajuntament de Tarragona,NaN,Government,Open Data Tarragona,city spain,NaN,...,Tarragona,"41.1157, 1.2496",ES,ca es en,active,NaN,NaN,NaN,NaN,NaN
4,ajuntament-de-terassa,Open Data Terassa,https://opendata.terrassa.cat/,Ajuntament de Terassa,Ajuntament de Terassa,NaN,Government,Open Data Terassa,city spain,NaN,...,Terrasa,"41.5611, 2.0081",ES,es en,active,NaN,NaN,NaN,NaN,NaN


In [4]:
print(df.shape)
print(df.columns)
print(df.isnull().sum())

(605, 22)
Index(['name', 'title', 'url', 'author', 'publisher', 'issued',
       'publisher_classification', 'description', 'tags', 'license_id',
       'license_url', 'license_notes', 'place', 'location', 'country',
       'language', 'status', 'metadatacreated', 'generator', 'api_endpoint',
       'api_type', 'full_metadata_download'],
      dtype='str')
name                          0
title                         0
url                           0
author                       65
publisher                    61
issued                      506
publisher_classification    383
description                  52
tags                         74
license_id                  315
license_url                 564
license_notes               582
place                         2
location                     23
country                      15
language                      9
status                        0
metadatacreated             143
generator                   508
api_endpoint                559
a

In [5]:
text_column = "description"   # change if needed

df = df.dropna(subset=[text_column])
df = df.reset_index(drop=True)

In [6]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)        # remove URLs
    text = re.sub(r"[^a-zA-Z\s]", "", text)    # remove punctuation/numbers

    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]

    return " ".join(tokens)

In [8]:
print(df.head())
print(df.columns.tolist())

                       name                            title  \
0                 a2gov_org              Ann Arbor, Michigan   
1  acikveri-sahinbey-bel-tr  Açık Veri Portali - Test Yayını   
2          africa_open_data                 Africa Open Data   
3   ajuntament-de-tarragona              Open Data Tarragona   
4     ajuntament-de-terassa                Open Data Terassa   

                                            url                   author  \
0  https://www.a2gov.org/services/data-catalog/        City of Ann Arbor   
1      https://acikveri.sahinbey.bel.tr/dataset                 pinardag   
2                   https://africaopendata.org/         Africa Open Data   
3               https://opendata.tarragona.cat/  Ajuntament de Tarragona   
4                https://opendata.terrassa.cat/    Ajuntament de Terassa   

                 publisher      issued publisher_classification  \
0        City of Ann Arbor         NaN                      NaN   
1      SahinBey Belediye

In [10]:
print(df.columns)

Index(['name', 'title', 'url', 'author', 'publisher', 'issued',
       'publisher_classification', 'description', 'tags', 'license_id',
       'license_url', 'license_notes', 'place', 'location', 'country',
       'language', 'status', 'metadatacreated', 'generator', 'api_endpoint',
       'api_type', 'full_metadata_download'],
      dtype='str')


In [16]:
print(df.dtypes)

name                        str
title                       str
url                         str
author                      str
publisher                   str
issued                      str
publisher_classification    str
description                 str
tags                        str
license_id                  str
license_url                 str
license_notes               str
place                       str
location                    str
country                     str
language                    str
status                      str
metadatacreated             str
generator                   str
api_endpoint                str
api_type                    str
full_metadata_download      str
dtype: object


In [19]:
print(type(X))
print(X.shape)
print(X.head())

<class 'pandas.DataFrame'>
(553, 0)
Empty DataFrame
Columns: []
Index: [0, 1, 2, 3, 4]


In [24]:
print(model)
print(type(model))
print(dir(model)[:30])

KMeans(n_clusters=4)
<class 'sklearn.cluster._kmeans.KMeans'>
['__abstractmethods__', '__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__sklearn_clone__', '__sklearn_tags__', '__slots__']


In [26]:
print(df.head())
print(df.columns)

                       name                            title  \
0                 a2gov_org              Ann Arbor, Michigan   
1  acikveri-sahinbey-bel-tr  Açık Veri Portali - Test Yayını   
2          africa_open_data                 Africa Open Data   
3   ajuntament-de-tarragona              Open Data Tarragona   
4     ajuntament-de-terassa                Open Data Terassa   

                                            url                   author  \
0  https://www.a2gov.org/services/data-catalog/        City of Ann Arbor   
1      https://acikveri.sahinbey.bel.tr/dataset                 pinardag   
2                   https://africaopendata.org/         Africa Open Data   
3               https://opendata.tarragona.cat/  Ajuntament de Tarragona   
4                https://opendata.terrassa.cat/    Ajuntament de Terassa   

                 publisher      issued publisher_classification  \
0        City of Ann Arbor         NaN                      NaN   
1      SahinBey Belediye

In [27]:
df.to_csv("portals_clustered.csv", index=False)
print("Saved successfully!")

Saved successfully!
